# Beyond the GPU: TPUs, LPUs and What Specialization Buys

The runnable code of [Beyond the GPU: TPUs, LPUs and What Specialization Buys](https://residual-stream.lvtan71.workers.dev/posts/beyond-the-gpu-tpus-and-lpus/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/beyond-the-gpu-tpus-and-lpus.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "beyond-the-gpu-tpus-and-lpus"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## The systolic array

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/beyond-the-gpu-tpus-and-lpus/#the-systolic-array)

In [ ]:
import numpy as np

# A 3x3 output-stationary systolic array computing C = A @ B, cycle by cycle.
# Cell (i, j) owns C[i, j]. Row i of A enters from the left edge and moves
# one cell right per cycle; column j of B enters from the top and moves one
# cell down per cycle. Each input is skewed by one cycle per row (column),
# so A[i, k] and B[k, j] meet in cell (i, j) at cycle i + j + k. Every cell
# multiplies what arrives, adds to its own total, and passes both inputs on:
# no cell ever reads memory.
n = 3
A = np.arange(1, 10).reshape(n, n)
B = np.arange(1, 10).reshape(n, n)[::-1]
C = np.zeros((n, n), dtype=int)
a_reg = np.zeros((n, n), dtype=int)       # the A value each cell holds this cycle
b_reg = np.zeros((n, n), dtype=int)
macs = 0
for cycle in range(3 * n - 2):
    # shift: A moves right, B moves down; the edges take the next skewed input
    a_reg[:, 1:] = a_reg[:, :-1].copy(); b_reg[1:, :] = b_reg[:-1, :].copy()
    for i in range(n):
        k = cycle - i
        a_reg[i, 0] = A[i, k] if 0 <= k < n else 0
    for j in range(n):
        k = cycle - j
        b_reg[0, j] = B[k, j] if 0 <= k < n else 0
    C += a_reg * b_reg
    active = int(((a_reg != 0) & (b_reg != 0)).sum())
    macs += active
    print(f"cycle {cycle}: {active} of {n * n} cells multiplying")
print(f"\nresult equals A @ B: {np.array_equal(C, A @ B)}   ({macs} multiply-adds in {3 * n - 2} cycles)")
print(C)

# Memory traffic: each of A's and B's n^2 values is read from memory once
# and then reused n times as it passes through the array.
for n in (3, 128, 256):
    print(f"n = {n:3d}: {n ** 3:>10,d} multiply-adds from {2 * n * n:>7,d} values read: "
          f"each value used {2 * n ** 3 // (2 * n * n):3d} times")

The post printed:

```text
cycle 0: 1 of 9 cells multiplying
cycle 1: 3 of 9 cells multiplying
cycle 2: 6 of 9 cells multiplying
cycle 3: 7 of 9 cells multiplying
cycle 4: 6 of 9 cells multiplying
cycle 5: 3 of 9 cells multiplying
cycle 6: 1 of 9 cells multiplying

result equals A @ B: True   (27 multiply-adds in 7 cycles)
[[ 18  24  30]
 [ 54  69  84]
 [ 90 114 138]]
n =   3:         27 multiply-adds from      18 values read: each value used   3 times
n = 128:  2,097,152 multiply-adds from  32,768 values read: each value used 128 times
n = 256: 16,777,216 multiply-adds from 131,072 values read: each value used 256 times
```

## The SRAM capacity problem

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/beyond-the-gpu-tpus-and-lpus/#the-sram-capacity-problem)

In [ ]:
import math
# Holding a 70B-parameter model, and the fastest one stream of tokens could
# go, on four kinds of chip (vendor specs). Decoding one token at batch 1
# reads every weight once, so a bound on its time is weight bytes divided by
# the memory bandwidth the weights sit behind, or FLOPs divided by compute,
# whichever is larger. Bounds, not measurements: they ignore the links
# between chips, the KV cache and any software overhead.
P = 70e9
chips = {   # name: (memory per chip in bytes, bandwidth to it in bytes/s, dense FLOP/s at 16-bit)
    "GroqChip (SRAM)":  (230e6,          80e12,   188e12),
    "H100 SXM (HBM)":   (80e9,           3.35e12, 989e12),
    "TPU v6e (HBM)":    (32e9,           1.638e12, 918e12),
    "TPU7x (HBM)":      (192 * 2 ** 30,  7.38e12, 2307e12),
}
for bytes_per_param, label in ((2, "16-bit weights"), (1, "8-bit weights")):
    W = P * bytes_per_param
    print(f"70B model, {label}: {W / 1e9:.0f} GB")
    print(f"  {'chip':18s} {'chips to hold it':>16s} {'tokens/s bound':>15s} {'on 8 chips':>11s}")
    for name, (mem, bw, flops) in chips.items():
        n = math.ceil(W / mem)
        bound = lambda k: 1 / max(W / (k * bw), 2 * P / (k * flops))
        if name.startswith("Groq"):
            # Layers are spread across chips and run one after another for a
            # given token: each chip reads only its own share, but in turn, so
            # one stream's time is all the bytes over one chip's bandwidth,
            # however many chips there are.
            print(f"  {name:18s} {n:16d} {1 / max(W / bw, 2 * P / flops):15.0f} {'-':>11s}")
        else:
            # Tensor parallelism: k chips each read their share at the same time.
            print(f"  {name:18s} {n:16d} {bound(n):15.0f} {bound(max(n, 8)):11.0f}")
    print()

The post printed:

```text
70B model, 16-bit weights: 140 GB
  chip               chips to hold it  tokens/s bound  on 8 chips
  GroqChip (SRAM)                 609             571           -
  H100 SXM (HBM)                    2              48         191
  TPU v6e (HBM)                     5              58          94
  TPU7x (HBM)                       1              53         422

70B model, 8-bit weights: 70 GB
  chip               chips to hold it  tokens/s bound  on 8 chips
  GroqChip (SRAM)                 305            1143           -
  H100 SXM (HBM)                    1              48         383
  TPU v6e (HBM)                     3              70         187
  TPU7x (HBM)                       1             105         843
```